# Experiments 55-62
**Prueba de hiperparámetros:** multi_scale / weight_decay / dropout / momentum 

- **Model:**
    1. `yolov8m` *(Medium)*
- **Dataset:** 3.5m | 90º ***(v4i)***
- **Crop Tiles:** 640px
- **Sizes:** small & mid
- **Experiments:**
    1. Multi Scale: `multi_scale=True`
    1. Weight Decay: `weight_decay=0.0001` | `multi_scale=True`
    1. Dropout: `dropout=0.2` | `multi_scale=True`
    1. Momentum: `momentum=0.95` | `multi_scale=True`
- **Reference:** Default parameters
    - `multi_scale=False`
    - `weight_decay=0.0005`
    - `dropout=0`
    - `momentum=0.937`

## Init

In [1]:
import os
import shutil
import fnmatch
import pickle

In [2]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 81.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 104.7 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling

## Helper Functions

In [3]:
# Change for different file formats
reference = {
  "small": {
    "suffix": ".S",
    "file": "209"
  },
  "mid": {
    "suffix": ".M",
    "file": "503"
  },
  "large": {
    "suffix": ".L",
    "file": "000"
  }
}

In [4]:
# Clone config files
def copy_config(src_folder, dest_folder):
    """
    Copies files from src_folder to dest_folder, excluding subfolders.

    Args:
        src_folder: The path to the source folder.
        dest_folder: The path to the destination folder.
    """

    try:
        # Ensure destination folder exists
        os.makedirs(dest_folder, exist_ok=True)

        for filename in os.listdir(src_folder):
            src_path = os.path.join(src_folder, filename)
            dest_path = os.path.join(dest_folder, filename)

            if os.path.isfile(src_path):
                shutil.copy2(src_path, dest_path) #copy metadata as well.
                #Use shutil.copy for not copying metadata.
                print(f"Copied: {filename}")
            #else: #optional
                #print(f"Skipped (not a file): {filename}") #optional. Uncomment if you want to see the skipped folders.

        print("✅ Copying complete.")

    except Exception as e:
        print(f"❌ An error occurred: {e}")


In [5]:
# Copy filtered dataset images/labels
def copy_and_filter_folder(src_folder, dest_folder, pattern):
    """
    Copies a folder and files that match the given pattern.
    Alerts the user when a folder or file already exists but *does not* overwrite.
    Creates only what is needed.

    :param src_folder: Path to the source folder.
    :param dest_folder: Path to the destination folder.
    :param pattern: Filename pattern to keep (e.g., "*.txt").
    """
    try:
        # Ensure destination folder exists
        if not os.path.exists(dest_folder):
            print(f"✓ Creating destination folder '{dest_folder}'.\n")
            os.makedirs(dest_folder)
        else:
            print(f"✓ Destination folder '{dest_folder}' already exists.\n")

        # Walk through the source folder
        for root, _, files in os.walk(src_folder):
            relative_path = os.path.relpath(root, src_folder)
            new_root = os.path.join(dest_folder, relative_path)

            if not os.path.exists(new_root):
                print(f"Creating subdirectory '{new_root}'")
                os.makedirs(new_root)
            else:
                print(f"❕Subdirectory '{new_root}' already exists.")
                print("Make sure the data inside is relevant. Otherwise, just delete the folder and repeat the cloning process.")

            for file in files:
                if fnmatch.fnmatch(file, pattern + "*"):
                    src_file = os.path.join(root, file)
                    dest_file = os.path.join(new_root, file)

                    if not os.path.exists(dest_file):
                        shutil.copy2(src_file, dest_file)  # copy metadata as well
                    else:
                        print(f"❗️File '{dest_file}' already exists. Skipping.")

            print(f" ✓ Copying files complete.\n")
        print("✅ Copying dataset complete.")

    except Exception as e:
        print(f"❌ An error occurred: {e}")

In [6]:
def copy_directory(source, destination, overwrite=True):
    try:
        if overwrite and os.path.exists(destination):
            shutil.rmtree(destination)  # Remove existing directory
        shutil.copytree(source, destination)
        print(f"Directory '{source}' copied to '{destination}'")
        return True  # Indicate success
    except FileExistsError:
        print(f"Destination '{destination}' exists. Use overwrite=True to replace.")
        return False  # Indicate failure

In [7]:
# Load/Download prediction results (BB + confidence) as .pkl file

def save_results(results, filename):
    with open(filename, 'wb') as f:
        pickle.dump(results, f)

def load_results(filename):
    with open(filename, 'rb') as f:
        return pickle.load(f)

In [8]:
def save_on_cloud(source: str, destination: str):
    """
    Saves a folder to a cloud storage location (e.g., Google Drive in Colab).

    Args:
        source (str): The path to the source folder.
        destination (str): The path to the destination folder (in cloud storage).
    """
    # 0. Input Validation (Assertions)
    assert isinstance(source, str), "Source must be a string."
    assert isinstance(destination, str), "Destination must be a string."

    try:
        # 1. Verify Source Folder
        if not os.path.exists(destination):
            os.makedirs(destination)

        # 2. Copy the Folder
        shutil.copytree(source, destination, dirs_exist_ok=True)
        print("✅ Folder copied successfully:\n  ",source,"\n  -->",destination)

    except Exception as e:
        print(f"❌ An error occurred: {e}")

### Validation functions

In [9]:
cm = lambda results: results.confusion_matrix.matrix.tolist() if hasattr(results, 'confusion_matrix') and hasattr(results.confusion_matrix, 'matrix') else None

In [10]:
import json

def save_json(results):
  try:
    data_to_store = {
        "confusion_matrix": cm(results),
        "results_dict": results.results_dict,
        "speed": results.speed
    }

    # Convert the Python dictionary to a JSON string
    json_data = json.dumps(data_to_store, indent=4)

    # You can now save this JSON string to a file
    folder = str(results.save_dir)
    with open(f"/content/{folder}/results.json", "w") as f:
        f.write(json_data)

    print("✅ JSON file stored in:", folder)

  except Exception as e:
      print(f"❌ An error occurred: {e}")

  #return json_data


In [11]:
def gimme_metrics(results):
  matrix = cm(results)
  total_det = sum(sum(value) for value in matrix)
  percentages = []
  for row in matrix:
      values_percentages = []
      for value in row:
          if total_det != 0:
              percentage = (value / total_det) * 100
          else:
              percentage = 0.0
          values_percentages.append(f"{percentage:.2f}%")
      percentages.append(values_percentages)

  print("Total objects detected:", total_det)
  print("Confusion matrix:")
  for row in percentages:
      print(row)

  return matrix


In [12]:
def show_cm(TP, FP, FN):
    matrix = [[TP, FP], [FN, 0]]
    total_det = sum(sum(value) for value in matrix)
    percentages = []
    for row in matrix:
        values_percentages = []
        for value in row:
            if total_det != 0:
                percentage = (value / total_det) * 100
            else:
                percentage = 0.0
            values_percentages.append(f"{percentage:.2f}%")
        percentages.append(values_percentages)

    print("Total objects detected:", total_det)
    print("\nConfusion matrix:")
    for row in percentages:
        a, b = row
        print(f"[ {a} , {b} ]")

In [13]:
def show_metrics(TP, FP, FN):
    show_cm(TP, FP, FN)
    accuracy = TP/(TP+FP+FN)
    precision = TP/(TP+FP)
    recall = TP/(TP+FN)
    f1 = 2 * (precision * recall) / (precision + recall)
    f2 = 1.25 * (precision * recall) / (0.25 * precision + recall)
    fm = (precision * recall) ** 0.5
    print("\nMetrics:")
    print(f"- Accuracy: {accuracy:.3f}")
    print(f"- Precision: {precision:.3f}")
    print(f"- Recall: {recall:.3f}")
    print(f"- F1 Score: {f1:.3f}")
    print(f"- F½ Score: {f2:.3f}")
    print(f"- G-mean: {fm:.3f}")

# Datasets builder

## Importing from Drive

In [16]:
!rm -rf /content/sample_data

In [17]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [18]:
# Check if the cloud path is ok and the dataset can be found
!ls /content/drive/MyDrive/YOLO

3.5m.v3i.yolov8.640px		       3.5m.v4i.yolov8_blended.640px
3.5m.v3i.yolov8.640px.aug.v1	       3.5m.v4i.yolov8_blended.640px_clahe
3.5m.v3i.yolov8.640px.aug.v1.soil_aug  best_e26.pt
3.5m.v3i.yolov8.640px_clahe	       Inference
3.5m.v3i.yolov8.640px.soil_aug	       models
3.5m.v4i.yolov8.640px		       optuna_yolov8_f1_study.db
3.5m.v4i.yolov8.640px_aug5m


In [19]:
drive_path = '/content/drive/MyDrive/YOLO'
drive_datasets_paths = os.listdir(drive_path)
drive_datasets = len(drive_datasets_paths)
if (drive_datasets) > 1:
    print("There are %d dataset options:" % drive_datasets)
else:
    print("Theres is only 1 dataset:")
drive_datasets_paths

There are 13 dataset options:


['3.5m.v3i.yolov8.640px',
 'Inference',
 'models',
 '3.5m.v3i.yolov8.640px.aug.v1',
 'best_e26.pt',
 'optuna_yolov8_f1_study.db',
 '3.5m.v3i.yolov8.640px.soil_aug',
 '3.5m.v3i.yolov8.640px.aug.v1.soil_aug',
 '3.5m.v3i.yolov8.640px_clahe',
 '3.5m.v4i.yolov8.640px',
 '3.5m.v4i.yolov8.640px_aug5m',
 '3.5m.v4i.yolov8_blended.640px',
 '3.5m.v4i.yolov8_blended.640px_clahe']

**For this experiments:** `3.5m.v4i.yolov8.640px`

In [20]:
choose_dataset = 10
index = choose_dataset - 1
model_name = os.listdir(drive_path)[index]
print("Chosen model:", model_name)

Chosen model: 3.5m.v4i.yolov8.640px


***Readme:***
*   **Option 1:** is desirable if you need to test many subset combinations in the same session (avoid downloading data twice from the cloud).
*   **Option 2:** is desired if you are goint to work just with one dataset  (avoid downloading unnecessary data from the cloud).
*   **Option 3:** is best if you're just going to test one subset combination  (avoid downloading any data from the cloud).



In [21]:
# Option 2 (download just the needed dataset)
cloud_path = f"/content/drive/MyDrive/YOLO/{model_name}/"
local_path = f"/content/YOLO/"
!mkdir $local_path
!cp -r $cloud_path $local_path

In [22]:
src_folder = f"/content/YOLO/{model_name}"
data = f"{src_folder}/data.yaml"
data

'/content/YOLO/3.5m.v4i.yolov8.640px/data.yaml'

## Download model

In [14]:
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [15]:
# Load pretrain YOLO v8 model
model = YOLO("yolov8m.pt")

100%|██████████| 49.7M/49.7M [00:00<00:00, 122MB/s]


# Experiments

### Optimization

In [38]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [41]:
# Garbage collection
import gc
torch.cuda.empty_cache()
gc.collect()

0

In [ ]:
# Reduce VRAM usage by reducing fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

### Info

In [ ]:
!nvidia-smi

Thu May  8 15:24:50 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8              9W /   70W |       2MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!yolo version

8.3.129


-----
## Experiment 55
### *YOLOv8 Mid | Multi Scale*
`multi_scale` *(Default: False)*: Booleano que activa el entrenamiento multi-escala, afectando el tamaño de las imágenes de entrada durante el entrenamiento. Este es un parámetro de entrenamiento.

### Train

In [ ]:
# Set's maximum training time (in hours)
time: float = 1 # Depending on remaining Colab cuota
# This overrides the epochs argument, allowing training to automaticallystop after the specified duration.
# Useful for time-constrained training scenarios. (aka Colab cuotas)

In [ ]:
# Train model
model = YOLO("yolov8m.pt")
history = model.train(
    data=data,
    epochs=500,
    val=True,
    imgsz=640,
    batch=-1,
    patience=100,
    time = time,
    multi_scale=True, # Desactivado por defecto
)

Ultralytics 8.3.129 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/YOLO/3.5m.v4i.yolov8.640px/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=True, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose

100%|██████████| 755k/755k [00:00<00:00, 41.3MB/s]

Overriding model.yaml nc=80 with nc=1

                   from  n    params  module                                       arguments                     
  0                  -1  1      1392  ultralytics.nn.modules.conv.Conv             [3, 48, 3, 2]                 
  1                  -1  1     41664  ultralytics.nn.modules.conv.Conv             [48, 96, 3, 2]                
  2                  -1  2    111360  ultralytics.nn.modules.block.C2f             [96, 96, 2, True]             
  3                  -1  1    166272  ultralytics.nn.modules.conv.Conv             [96, 192, 3, 2]               
  4                  -1  4    813312  ultralytics.nn.modules.block.C2f             [192, 192, 4, True]           
  5                  -1  1    664320  ultralytics.nn.modules.conv.Conv             [192, 384, 3, 2]              
  6                  -1  4   3248640  ultralytics.nn.modules.block.C2f             [384, 384, 4, True]           
  7                  -1  1   1991808  ultralytics

 21                  -1  2   4207104  ultralytics.nn.modules.block.C2f             [960, 576, 2]                 
 22        [15, 18, 21]  1   3776275  ultralytics.nn.modules.head.Detect           [1, [192, 384, 576]]          
Model summary: 169 layers, 25,856,899 parameters, 25,856,883 gradients, 79.1 GFLOPs

Transferred 469/475 items from pretrained weights
Freezing layer 'model.22.dfl.conv.weight'
AMP: running Automatic Mixed Precision (AMP) checks...


100%|██████████| 5.35M/5.35M [00:00<00:00, 149MB/s]


...

    165/226      12.1G       1.45     0.8171      1.137         85        448: 100%|██████████| 15/15 [00:10<00:00,  1.46it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:02<00:00,  1.29it/s]

                   all        108       3472      0.551      0.492      0.481      0.161
EarlyStopping: Training stopped early as no improvement observed in last 100 epochs. Best results observed at epoch 65, best model saved as best.pt.
To update EarlyStopping(patience=100) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.



165 epochs completed in 0.733 hours.
Optimizer stripped from runs/detect/train/weights/last.pt, 52.0MB
Optimizer stripped from runs/detect/train/weights/best.pt, 52.0MB

Validating runs/detect/train/weights/best.pt...
Ultralytics 8.3.129 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:05<00:00,  1.77s/it]


                   all        108       3472      0.546      0.478      0.481      0.172
Speed: 0.2ms preprocess, 11.4ms inference, 0.0ms loss, 4.1ms postprocess per image
Results saved to runs/detect/train


In [ ]:
history

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b8bb5903490>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [ ]:
# Show the hyperparameters set
model.trainer.validator.args

namespace(task='detect',
          mode='train',
          model='yolov8m.pt',
          data='/content/YOLO/3.5m.v4i.yolov8.640px/data.yaml',
          epochs=500,
          time=1,
          patience=100,
          batch=19,
          imgsz=640,
          save=True,
          save_period=-1,
          cache=False,
          device=None,
          workers=8,
          project=None,
          name='train',
          exist_ok=False,
          pretrained=True,
          optimizer='auto',
          verbose=True,
          seed=0,
          deterministic=True,
          single_cls=False,
          rect=False,
          cos_lr=False,
          close_mosaic=10,
          resume=False,
          amp=True,
          fraction=1.0,
          profile=False,
          freeze=None,
          multi_scale=True,
          overlap_mask=True,
          mask_ratio=4,
          dropout=0.0,
          val=True,
          split='val',
          save_json=False,
          conf=0.001,
          iou=0.7,
     

In [ ]:
print(f"Saved into: {history.save_dir}")

Saved into: runs/detect/train


### Validation

In [ ]:
# Load currently trained YOLO model
model = YOLO(f"/content/{history.save_dir}/weights/best.pt")

In [ ]:
# Validate the model
results = model.val(data=data,
          batch=64,
          conf=0.224, # best values found with optuna
          iou=0.57, # best values found with optuna
          verbose=True,
          save_json=True)

Ultralytics 8.3.129 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1454.0±532.0 MB/s, size: 94.6 KB)


val: Scanning /content/YOLO/3.5m.v4i.yolov8.640px/valid/labels.cache... 108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 108/108 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:06<00:00,  3.26s/it]


                   all        108       3472       0.58      0.486      0.522      0.205
Speed: 5.3ms preprocess, 22.1ms inference, 0.0ms loss, 2.5ms postprocess per image
Saving runs/detect/val2/predictions.json...
Results saved to runs/detect/val2


In [ ]:
print(f"Saved into: {results.save_dir}")

Saved into: runs/detect/val2


In [ ]:
save_json(results)

✅ JSON file stored in: runs/detect/val2


In [ ]:
matrix = gimme_metrics(results)

Total objects detected: 4496.0
Confusion matrix:
['41.86%', '22.78%']
['35.36%', '0.00%']


### Save results

In [ ]:
# Store model weights and metrics
save_on_cloud(source='/content/runs/', destination='/content/drive/MyDrive/save1/')

✅ Folder copied successfully:
   /content/runs/ 
  --> /content/drive/MyDrive/save1/


### Metrics

In [ ]:
matrix

[[1882.0, 1024.0], [1590.0, 0.0]]

In [ ]:
# Setting values from CM graph
TP = matrix[0][0]
FP = matrix[0][1]
FN = matrix[1][0]
# Confusion matrix
show_metrics(TP, FP, FN)

Total objects detected: 4496.0

Confusion matrix:
[ 41.86% , 22.78% ]
[ 35.36% , 0.00% ]

Metrics:
- Accuracy: 0.419
- Precision: 0.648
- Recall: 0.542
- F1 Score: 0.590
- F½ Score: 0.623
- G-mean: 0.592


Comparación con Reference:
- Accuracy: Mejora (0.419 > 0.406)
- Precision: Mejora (0.648 > 0.579)
- Recall: No mejora (0.542 < 0.576)
- F1 Score: Mejora (0.590 > 0.577)
- F½ Score: Mejora (0.623 > 0.578)
- G-mean: Mejora (0.592 > 0.577)

-----
## Experiment 56
### *YOLOv8 Mid | Weight Decay (L2 reg)*
`weight_decay` *(`Default: 0.0005`)*: Float para la regularización L2. Penaliza pesos grandes para prevenir overfitting. Afecta la función de pérdida y el entrenamiento. Este es un parámetro de entrenamiento. Con valores bajos (menores a 0.0005) ayuda a mitigar el subajuste (underfitting) que podría ser causado por una regularización excesiva.

- Se observó que el modelo mejoraba con la aplicación de multi_scale, por lo que se conservó ese hiperparámetro para el nuevo experimento.

### Train

In [ ]:
# Set's maximum training time (in hours)
time: float = 1 # Depending on remaining Colab cuota
# This overrides the epochs argument, allowing training to automaticallystop after the specified duration.
# Useful for time-constrained training scenarios. (aka Colab cuotas)

In [ ]:
# Train model
model = YOLO("yolov8m.pt")
history = model.train(
    data=data,
    epochs=500,
    val=True,
    imgsz=640,
    batch=-1,
    patience=100,
    time = time,
    multi_scale=True,
    weight_decay=0.0001, # Inferior a default
)

Ultralytics 8.3.129 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/YOLO/3.5m.v4i.yolov8.640px/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=True, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pos

train: Scanning /content/YOLO/3.5m.v4i.yolov8.640px/train/labels.cache... 270 images, 0 backgrounds, 0 corrupt: 100%|██████████| 270/270 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
AutoBatch: Computing optimal batch size for imgsz=640 at 60.0% CUDA memory utilization.
AutoBatch: CUDA:0 (Tesla T4) 14.74G total, 0.35G reserved, 0.32G allocated, 14.07G free
      Params      GFLOPs  GPU_mem (GB)  forward (ms) backward (ms)                   input                  output


    25856899       79.07         1.724         52.38         61.18        (1, 3, 640, 640)                    list
    25856899       158.1         2.261         58.33         69.82        (2, 3, 640, 640)                    list
    25856899       316.3         3.118            47          86.1        (4, 3, 640, 640)                    list
    25856899       632.5         4.679         81.73         139.4        (8, 3, 640, 640)                    list
    25856899        1265         7.743         154.8         268.7       (16, 3, 640, 640)                    list
AutoBatch: Using batch-size 16 for CUDA:0 8.46G/14.74G (57%) ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2009.1±566.2 MB/s, size: 89.0 KB)


train: Scanning /content/YOLO/3.5m.v4i.yolov8.640px/train/labels.cache... 270 images, 0 backgrounds, 0 corrupt: 100%|██████████| 270/270 [00:00<?, ?it/s]

...

    176/267      11.8G      1.442      0.762      1.091        447        800: 100%|██████████| 17/17 [00:09<00:00,  1.81it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:01<00:00,  2.01it/s]

                   all        108       3472      0.535      0.469      0.436      0.146
EarlyStopping: Training stopped early as no improvement observed in last 100 epochs. Best results observed at epoch 76, best model saved as best.pt.
To update EarlyStopping(patience=100) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.



176 epochs completed in 0.659 hours.
Optimizer stripped from runs/detect/train2/weights/last.pt, 52.0MB
Optimizer stripped from runs/detect/train2/weights/best.pt, 52.0MB

Validating runs/detect/train2/weights/best.pt...
Ultralytics 8.3.129 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:05<00:00,  1.49s/it]


                   all        108       3472       0.56      0.486      0.491      0.175
Speed: 1.1ms preprocess, 12.1ms inference, 0.0ms loss, 4.4ms postprocess per image
Results saved to runs/detect/train2


In [ ]:
history

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b8be0303490>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [ ]:
# Show the hyperparameters set
model.trainer.validator.args

namespace(task='detect',
          mode='train',
          model='yolov8m.pt',
          data='/content/YOLO/3.5m.v4i.yolov8.640px/data.yaml',
          epochs=500,
          time=1,
          patience=100,
          batch=16,
          imgsz=640,
          save=True,
          save_period=-1,
          cache=False,
          device=None,
          workers=8,
          project=None,
          name='train2',
          exist_ok=False,
          pretrained=True,
          optimizer='auto',
          verbose=True,
          seed=0,
          deterministic=True,
          single_cls=False,
          rect=False,
          cos_lr=False,
          close_mosaic=10,
          resume=False,
          amp=True,
          fraction=1.0,
          profile=False,
          freeze=None,
          multi_scale=True,
          overlap_mask=True,
          mask_ratio=4,
          dropout=0.0,
          val=True,
          split='val',
          save_json=False,
          conf=0.001,
          iou=0.7,
    

In [ ]:
print(f"Saved into: {history.save_dir}")

Saved into: runs/detect/train2


### Validation

In [ ]:
# Load currently trained YOLO model
model = YOLO(f"/content/{history.save_dir}/weights/best.pt")

In [ ]:
# Validate the model
results = model.val(data=data,
          batch=64,
          conf=0.224, # best values found with optuna
          iou=0.57, # best values found with optuna
          verbose=True,
          save_json=True)

Ultralytics 8.3.129 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2669.0±659.3 MB/s, size: 104.9 KB)


val: Scanning /content/YOLO/3.5m.v4i.yolov8.640px/valid/labels.cache... 108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 108/108 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:06<00:00,  3.07s/it]


                   all        108       3472       0.58      0.497      0.528      0.208
Speed: 0.2ms preprocess, 26.1ms inference, 0.0ms loss, 2.2ms postprocess per image
Saving runs/detect/val3/predictions.json...
Results saved to runs/detect/val3


In [ ]:
print(f"Saved into: {results.save_dir}")

Saved into: runs/detect/val3


In [ ]:
save_json(results)

✅ JSON file stored in: runs/detect/val3


In [ ]:
matrix = gimme_metrics(results)

Total objects detected: 4549.0
Confusion matrix:
['41.75%', '23.68%']
['34.58%', '0.00%']


### Save results

In [ ]:
# Store model weights and metrics
save_on_cloud(source='/content/runs/', destination='/content/drive/MyDrive/save2/')

✅ Folder copied successfully:
   /content/runs/ 
  --> /content/drive/MyDrive/save2/


### Metrics

In [ ]:
matrix

[[1899.0, 1077.0], [1573.0, 0.0]]

In [ ]:
# Setting values from CM graph
TP = matrix[0][0]
FP = matrix[0][1]
FN = matrix[1][0]
# Confusion matrix
show_metrics(TP, FP, FN)

Total objects detected: 4549.0

Confusion matrix:
[ 41.75% , 23.68% ]
[ 34.58% , 0.00% ]

Metrics:
- Accuracy: 0.417
- Precision: 0.638
- Recall: 0.547
- F1 Score: 0.589
- F½ Score: 0.618
- G-mean: 0.591


Comparación con Reference:
- Accuracy: Mejora (0.417 > 0.406)
- Precision: Mejora (0.638 > 0.579)
- Recall: Constante (0.547 ~ 0.576)
- F1 Score: Constante (0.589 ~ 0.577)
- F½ Score: Mejora (0.618 > 0.578)
- G-mean: Mejora (0.591 > 0.577)

-----
## Experiment 57
### *YOLOv8 Mid | Dropout*
`dropout`  *(Default: 0)*: Float que añade regularización (dropout) en la capa de clasificación. Este es un parámetro que afecta la arquitectura/entrenamiento del modelo. Este es un parámetro de entrenamiento.

- Si bien la implementación de L2 reg representó una leve mejora respecto a la referencia, no fue mejor que el exp. 58. Por este motivo, se elimina este hiperparámetro siguiendo la premisa previa.

### Train

In [ ]:
# Set's maximum training time (in hours)
time: float = 1 # Depending on remaining Colab cuota
# This overrides the epochs argument, allowing training to automaticallystop after the specified duration.
# Useful for time-constrained training scenarios. (aka Colab cuotas)

In [ ]:
# Train model
model = YOLO("yolov8m.pt")
history = model.train(
    data=data,
    epochs=500,
    val=True,
    imgsz=640,
    batch=-1,
    patience=100,
    time = time,
    multi_scale=True,
    #weight_decay=0.0001,
    dropout=0.2, # Superior a default
)

Ultralytics 8.3.129 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/YOLO/3.5m.v4i.yolov8.640px/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.2, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=True, name=train3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pos

train: Scanning /content/YOLO/3.5m.v4i.yolov8.640px/train/labels.cache... 270 images, 0 backgrounds, 0 corrupt: 100%|██████████| 270/270 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
AutoBatch: Computing optimal batch size for imgsz=640 at 60.0% CUDA memory utilization.
AutoBatch: CUDA:0 (Tesla T4) 14.74G total, 5.61G reserved, 0.91G allocated, 8.23G free
      Params      GFLOPs  GPU_mem (GB)  forward (ms) backward (ms)                   input                  output


    25856899       79.07         1.812         41.69         39.69        (1, 3, 640, 640)                    list
    25856899       158.1         2.284         35.38         47.24        (2, 3, 640, 640)                    list
    25856899       316.3         3.160         41.75         76.09        (4, 3, 640, 640)                    list
    25856899       632.5         4.763         80.92         140.6        (8, 3, 640, 640)                    list
    25856899        1265         7.787         155.2         269.1       (16, 3, 640, 640)                    list
AutoBatch: Using batch-size 8 for CUDA:0 11.19G/14.74G (76%) ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1643.2±346.2 MB/s, size: 89.0 KB)


train: Scanning /content/YOLO/3.5m.v4i.yolov8.640px/train/labels.cache... 270 images, 0 backgrounds, 0 corrupt: 100%|██████████| 270/270 [00:00<?, ?it/s]

...

                   all        108       3472      0.539      0.469      0.459      0.156

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


    159/263      7.11G      1.561      0.847      1.164        298        704: 100%|██████████| 34/34 [00:11<00:00,  3.05it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 7/7 [00:01<00:00,  4.25it/s]


                   all        108       3472      0.533      0.466       0.45       0.15
EarlyStopping: Training stopped early as no improvement observed in last 100 epochs. Best results observed at epoch 59, best model saved as best.pt.
To update EarlyStopping(patience=100) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

159 epochs completed in 0.607 hours.
Optimizer stripped from runs/detect/train3/weights/last.pt, 52.0MB
Optimizer stripped from runs/detect/train3/weights/best.pt, 52.0MB

Validating runs/detect/train3/weights/best.pt...
Ultralytics 8.3.129 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 7/7 [00:07<00:00,  1.13s/it]


                   all        108       3472      0.537       0.48      0.486      0.178
Speed: 0.8ms preprocess, 11.0ms inference, 0.0ms loss, 10.3ms postprocess per image
Results saved to runs/detect/train3


In [ ]:
history

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b8be04f5d10>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [ ]:
# Show the hyperparameters set
model.trainer.validator.args

namespace(task='detect',
          mode='train',
          model='yolov8m.pt',
          data='/content/YOLO/3.5m.v4i.yolov8.640px/data.yaml',
          epochs=500,
          time=1,
          patience=100,
          batch=8,
          imgsz=640,
          save=True,
          save_period=-1,
          cache=False,
          device=None,
          workers=8,
          project=None,
          name='train3',
          exist_ok=False,
          pretrained=True,
          optimizer='auto',
          verbose=True,
          seed=0,
          deterministic=True,
          single_cls=False,
          rect=False,
          cos_lr=False,
          close_mosaic=10,
          resume=False,
          amp=True,
          fraction=1.0,
          profile=False,
          freeze=None,
          multi_scale=True,
          overlap_mask=True,
          mask_ratio=4,
          dropout=0.2,
          val=True,
          split='val',
          save_json=False,
          conf=0.001,
          iou=0.7,
     

In [ ]:
print(f"Saved into: {history.save_dir}")

Saved into: runs/detect/train3


### Validation

In [ ]:
# Load currently trained YOLO model
model = YOLO(f"/content/{history.save_dir}/weights/best.pt")

In [ ]:
# Validate the model
results = model.val(data=data,
          batch=64,
          conf=0.224, # best values found with optuna
          iou=0.57, # best values found with optuna
          verbose=True,
          save_json=True)

Ultralytics 8.3.129 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2240.7±451.4 MB/s, size: 97.1 KB)


val: Scanning /content/YOLO/3.5m.v4i.yolov8.640px/valid/labels.cache... 108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 108/108 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:06<00:00,  3.36s/it]


                   all        108       3472      0.582      0.488      0.527      0.215
Speed: 7.3ms preprocess, 21.7ms inference, 0.0ms loss, 3.2ms postprocess per image
Saving runs/detect/val4/predictions.json...
Results saved to runs/detect/val4


In [ ]:
print(f"Saved into: {results.save_dir}")

Saved into: runs/detect/val4


In [ ]:
save_json(results)

✅ JSON file stored in: runs/detect/val4


In [ ]:
matrix = gimme_metrics(results)

Total objects detected: 4509.0
Confusion matrix:
['41.56%', '23.00%']
['35.44%', '0.00%']


### Save results

In [ ]:
# Store model weights and metrics
save_on_cloud(source='/content/runs/', destination='/content/drive/MyDrive/save3/')

✅ Folder copied successfully:
   /content/runs/ 
  --> /content/drive/MyDrive/save3/


### Metrics

In [ ]:
matrix

[[1874.0, 1037.0], [1598.0, 0.0]]

In [ ]:
# Setting values from CM graph
TP = matrix[0][0]
FP = matrix[0][1]
FN = matrix[1][0]
# Confusion matrix
show_metrics(TP, FP, FN)

Total objects detected: 4509.0

Confusion matrix:
[ 41.56% , 23.00% ]
[ 35.44% , 0.00% ]

Metrics:
- Accuracy: 0.416
- Precision: 0.644
- Recall: 0.540
- F1 Score: 0.587
- F½ Score: 0.620
- G-mean: 0.589


Comparación con Reference:
- Accuracy: Mejora (0.416 > 0.406)
- Precision: Mejora (0.644 > 0.579)
- Recall: No mejora (0.540 < 0.576)
- F1 Score: Constante (0.587 ~ 0.577)
- F½ Score: Mejora (0.620 > 0.578)
- G-mean: Mejora (0.589 > 0.577)

-----
## Experiment 58
### *YOLOv8 Mid | Momentum*
`momentum` *(Default: 0.937)*: Float para el optimizador (SGD o Adam). Afecta cómo se actualizan los pesos durante el entrenamiento. Este es un parámetro de entrenamiento.

- No se observó una mejora del modelo al agregar dropout.

### Train

In [ ]:
# Set's maximum training time (in hours)
time: float = 1 # Depending on remaining Colab cuota
# This overrides the epochs argument, allowing training to automaticallystop after the specified duration.
# Useful for time-constrained training scenarios. (aka Colab cuotas)

In [ ]:
# Train model
model = YOLO("yolov8m.pt")
history = model.train(
    data=data,
    epochs=500,
    val=True,
    imgsz=640,
    batch=-1,
    patience=100,
    time = time,
    multi_scale=True,
    #weight_decay=0.0001,
    #dropout=0.2,
    momentum=0.95, # Superior a default
)

Metrics:

- Accuracy: 0.426
- Precision: 0.630
- Recall: 0.569
- F1 Score: 0.598
- F½ Score: 0.617
- G-mean: 0.599

Comparación con Reference:
- Accuracy: Mejora (0.416 vs 0.406)
- Precision: Mejora (0.644 vs 0.579)
- Recall: Empeora (0.540 vs 0.576)
- F1 Score: Constante (0.587 vs 0.577)
- F½ Score: Mejora (0.620 vs 0.578)
- G-mean: Mejora (0.589 vs 0.577)

# Mixes

## Experiment 59
### *YOLOv8 Mid | Mix 1*
Se prueba con un dropout mayor sumado a momentum y multi_scale.

### Train

In [ ]:
# Set's maximum training time (in hours)
time: float = 1 # Depending on remaining Colab cuota
# This overrides the epochs argument, allowing training to automaticallystop after the specified duration.
# Useful for time-constrained training scenarios. (aka Colab cuotas)

In [ ]:
# Train model
model = YOLO("yolov8m.pt")
history = model.train(
    data=data,
    epochs=500,
    val=True,
    imgsz=640,
    batch=-1,
    patience=100,
    time = time,
    multi_scale=True,
    #weight_decay=0.0001,
    dropout=0.4, # Superior al anterior
    momentum=0.95, # Superior a default
)

Ultralytics 8.3.129 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/YOLO/3.5m.v4i.yolov8.640px/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.4, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.95, mosaic=1.0, multi_scale=True, name=train5, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose

train: Scanning /content/YOLO/3.5m.v4i.yolov8.640px/train/labels.cache... 270 images, 0 backgrounds, 0 corrupt: 100%|██████████| 270/270 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
AutoBatch: Computing optimal batch size for imgsz=640 at 60.0% CUDA memory utilization.
AutoBatch: CUDA:0 (Tesla T4) 14.74G total, 11.16G reserved, 1.25G allocated, 2.33G free
      Params      GFLOPs  GPU_mem (GB)  forward (ms) backward (ms)                   input                  output


    25856899       79.07         3.079         33.23         27.25        (1, 3, 640, 640)                    list
    25856899       158.1         3.509         34.13         40.91        (2, 3, 640, 640)                    list
    25856899       316.3         4.343         39.43         74.37        (4, 3, 640, 640)                    list
    25856899       632.5         5.924         80.32         134.5        (8, 3, 640, 640)                    list
    25856899        1265         8.967         151.5           260       (16, 3, 640, 640)                    list
WARNING ⚠️ AutoBatch: batch=-4 outside safe range, using default batch-size 16.
AutoBatch: Using batch-size 16 for CUDA:0 21.41G/14.74G (145%) ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1684.2±330.3 MB/s, size: 89.0 KB)


train: Scanning /content/YOLO/3.5m.v4i.yolov8.640px/train/labels.cache... 270 images, 0 backgrounds, 0 corrupt: 100%|██████████| 270/270 [00:00<?, ?it/s]

...


      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


    154/221      13.5G      1.512     0.8322      1.125        409        352: 100%|██████████| 17/17 [00:10<00:00,  1.62it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:01<00:00,  2.46it/s]


                   all        108       3472       0.55      0.502      0.483      0.162
EarlyStopping: Training stopped early as no improvement observed in last 100 epochs. Best results observed at epoch 54, best model saved as best.pt.
To update EarlyStopping(patience=100) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

154 epochs completed in 0.696 hours.
Optimizer stripped from runs/detect/train5/weights/last.pt, 52.0MB
Optimizer stripped from runs/detect/train5/weights/best.pt, 52.0MB

Validating runs/detect/train5/weights/best.pt...
Ultralytics 8.3.129 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:05<00:00,  1.50s/it]


                   all        108       3472      0.554      0.495      0.504      0.177
Speed: 0.2ms preprocess, 11.2ms inference, 0.0ms loss, 4.0ms postprocess per image
Results saved to runs/detect/train5


In [ ]:
history

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b8ba4f05c50>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [ ]:
# Show the hyperparameters set
model.trainer.validator.args

namespace(task='detect',
          mode='train',
          model='yolov8m.pt',
          data='/content/YOLO/3.5m.v4i.yolov8.640px/data.yaml',
          epochs=500,
          time=1,
          patience=100,
          batch=16,
          imgsz=640,
          save=True,
          save_period=-1,
          cache=False,
          device=None,
          workers=8,
          project=None,
          name='train5',
          exist_ok=False,
          pretrained=True,
          optimizer='auto',
          verbose=True,
          seed=0,
          deterministic=True,
          single_cls=False,
          rect=False,
          cos_lr=False,
          close_mosaic=10,
          resume=False,
          amp=True,
          fraction=1.0,
          profile=False,
          freeze=None,
          multi_scale=True,
          overlap_mask=True,
          mask_ratio=4,
          dropout=0.4,
          val=True,
          split='val',
          save_json=False,
          conf=0.001,
          iou=0.7,
    

In [ ]:
print(f"Saved into: {history.save_dir}")

Saved into: runs/detect/train5


### Validation

In [ ]:
# Load currently trained YOLO model
model = YOLO(f"/content/{history.save_dir}/weights/best.pt")

In [ ]:
# Validate the model
results = model.val(data=data,
          batch=64,
          conf=0.224, # best values found with optuna
          iou=0.57, # best values found with optuna
          verbose=True,
          save_json=True)

Ultralytics 8.3.129 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2003.6±584.7 MB/s, size: 89.5 KB)


val: Scanning /content/YOLO/3.5m.v4i.yolov8.640px/valid/labels.cache... 108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 108/108 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:06<00:00,  3.14s/it]


                   all        108       3472       0.55      0.537      0.541      0.207
Speed: 5.2ms preprocess, 22.2ms inference, 0.0ms loss, 3.4ms postprocess per image
Saving runs/detect/val/predictions.json...
Results saved to runs/detect/val


In [ ]:
print(f"Saved into: {results.save_dir}")

Saved into: runs/detect/val


In [ ]:
save_json(results)

✅ JSON file stored in: runs/detect/val


In [ ]:
matrix = gimme_metrics(results)

Total objects detected: 4791.0
Confusion matrix:
['43.29%', '27.53%']
['29.18%', '0.00%']


### Save results

In [ ]:
# Store model weights and metrics
save_on_cloud(source='/content/runs/', destination='/content/drive/MyDrive/save5/')

✅ Folder copied successfully:
   /content/runs/ 
  --> /content/drive/MyDrive/save5/


### Metrics

In [ ]:
matrix

[[2074.0, 1319.0], [1398.0, 0.0]]

In [ ]:
# Setting values from CM graph
TP = matrix[0][0]
FP = matrix[0][1]
FN = matrix[1][0]
# Confusion matrix
show_metrics(TP, FP, FN)

Total objects detected: 4791.0

Confusion matrix:
[ 43.29% , 27.53% ]
[ 29.18% , 0.00% ]

Metrics:
- Accuracy: 0.433
- Precision: 0.611
- Recall: 0.597
- F1 Score: 0.604
- F½ Score: 0.608
- G-mean: 0.604


Comparación con Reference:
- Accuracy: Mejora (0.433 > 0.406)
- Precision: Mejora (0.611 > 0.579)
- Recall: Mejora (0.597 > 0.576)
- F1 Score: Mejora (0.604 > 0.577)
- F½ Score: Mejora (0.608 > 0.578)
- G-mean: Mejora (0.604 > 0.577)

-----
## Experiment 60
### *YOLOv8 Mid | Mix 2*

### Train

In [ ]:
# Set's maximum training time (in hours)
time: float = 1 # Depending on remaining Colab cuota
# This overrides the epochs argument, allowing training to automaticallystop after the specified duration.
# Useful for time-constrained training scenarios. (aka Colab cuotas)

In [ ]:
# Train model
model = YOLO("yolov8m.pt")
history = model.train(
    data=data,
    epochs=500,
    val=True,
    imgsz=640,
    batch=-1,
    patience=100,
    time = time,
    multi_scale=True,
    weight_decay=0.001, # Superior a default
    #dropout=0.2,
    momentum=0.98, # Superior al anterior
)

Ultralytics 8.3.129 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/YOLO/3.5m.v4i.yolov8.640px/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.98, mosaic=1.0, multi_scale=True, name=train7, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose

train: Scanning /content/YOLO/3.5m.v4i.yolov8.640px/train/labels.cache... 270 images, 0 backgrounds, 0 corrupt: 100%|██████████| 270/270 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
AutoBatch: Computing optimal batch size for imgsz=640 at 60.0% CUDA memory utilization.
AutoBatch: CUDA:0 (Tesla T4) 14.74G total, 1.34G reserved, 0.80G allocated, 12.60G free
      Params      GFLOPs  GPU_mem (GB)  forward (ms) backward (ms)                   input                  output


    25856899       79.07         3.097         42.91         45.06        (1, 3, 640, 640)                    list
    25856899       158.1         3.569         36.33          75.1        (2, 3, 640, 640)                    list
    25856899       316.3         4.408         44.96         76.92        (4, 3, 640, 640)                    list
    25856899       632.5         5.985         79.68         135.2        (8, 3, 640, 640)                    list
    25856899        1265         9.011           152         259.4       (16, 3, 640, 640)                    list
AutoBatch: Using batch-size 13 for CUDA:0 10.02G/14.74G (68%) ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1507.0±456.4 MB/s, size: 89.0 KB)


train: Scanning /content/YOLO/3.5m.v4i.yolov8.640px/train/labels.cache... 270 images, 0 backgrounds, 0 corrupt: 100%|██████████| 270/270 [00:00<?, ?it/s]

...

    184/217      11.4G      1.336      0.735        1.1        207        608: 100%|██████████| 21/21 [00:10<00:00,  1.95it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 5/5 [00:02<00:00,  2.21it/s]

                   all        108       3472      0.556      0.491      0.477      0.165


EarlyStopping: Training stopped early as no improvement observed in last 100 epochs. Best results observed at epoch 84, best model saved as best.pt.
To update EarlyStopping(patience=100) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

184 epochs completed in 0.852 hours.
Optimizer stripped from runs/detect/train7/weights/last.pt, 52.0MB
Optimizer stripped from runs/detect/train7/weights/best.pt, 52.0MB

Validating runs/detect/train7/weights/best.pt...
Ultralytics 8.3.129 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 5/5 [00:06<00:00,  1.37s/it]


                   all        108       3472      0.558      0.487        0.5      0.178
Speed: 0.8ms preprocess, 12.4ms inference, 0.0ms loss, 9.1ms postprocess per image
Results saved to runs/detect/train7


In [ ]:
history

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b8be0888c50>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [ ]:
# Show the hyperparameters set
model.trainer.validator.args

namespace(task='detect',
          mode='train',
          model='yolov8m.pt',
          data='/content/YOLO/3.5m.v4i.yolov8.640px/data.yaml',
          epochs=500,
          time=1,
          patience=100,
          batch=13,
          imgsz=640,
          save=True,
          save_period=-1,
          cache=False,
          device=None,
          workers=8,
          project=None,
          name='train7',
          exist_ok=False,
          pretrained=True,
          optimizer='auto',
          verbose=True,
          seed=0,
          deterministic=True,
          single_cls=False,
          rect=False,
          cos_lr=False,
          close_mosaic=10,
          resume=False,
          amp=True,
          fraction=1.0,
          profile=False,
          freeze=None,
          multi_scale=True,
          overlap_mask=True,
          mask_ratio=4,
          dropout=0.0,
          val=True,
          split='val',
          save_json=False,
          conf=0.001,
          iou=0.7,
    

In [ ]:
print(f"Saved into: {history.save_dir}")

Saved into: runs/detect/train7


### Validation

In [ ]:
# Load currently trained YOLO model
model = YOLO(f"/content/{history.save_dir}/weights/best.pt")

In [ ]:
# Validate the model
results = model.val(data=data,
          batch=64,
          conf=0.224, # best values found with optuna
          iou=0.57, # best values found with optuna
          verbose=True,
          save_json=True)

Ultralytics 8.3.129 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1759.6±317.9 MB/s, size: 89.0 KB)


val: Scanning /content/YOLO/3.5m.v4i.yolov8.640px/valid/labels.cache... 108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 108/108 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:06<00:00,  3.38s/it]


                   all        108       3472      0.595      0.493      0.536      0.214
Speed: 7.2ms preprocess, 21.8ms inference, 0.0ms loss, 2.8ms postprocess per image
Saving runs/detect/val6/predictions.json...
Results saved to runs/detect/val6


In [ ]:
print(f"Saved into: {results.save_dir}")

Saved into: runs/detect/val6


In [ ]:
save_json(results)

✅ JSON file stored in: runs/detect/val6


In [ ]:
matrix = gimme_metrics(results)

Total objects detected: 4453.0
Confusion matrix:
['42.60%', '22.03%']
['35.37%', '0.00%']


### Save results

In [ ]:
# Store model weights and metrics
save_on_cloud(source='/content/runs/', destination='/content/drive/MyDrive/save6/')

✅ Folder copied successfully:
   /content/runs/ 
  --> /content/drive/MyDrive/save6/


### Metrics

In [ ]:
matrix

[[1897.0, 981.0], [1575.0, 0.0]]

In [ ]:
# Setting values from CM graph
TP = matrix[0][0]
FP = matrix[0][1]
FN = matrix[1][0]
# Confusion matrix
show_metrics(TP, FP, FN)

Total objects detected: 4453.0

Confusion matrix:
[ 42.60% , 22.03% ]
[ 35.37% , 0.00% ]

Metrics:
- Accuracy: 0.426
- Precision: 0.659
- Recall: 0.546
- F1 Score: 0.597
- F½ Score: 0.633
- G-mean: 0.600


Comparación con Reference:
- Accuracy: Mejora (0.426 > 0.406)
- Precision: Mejora (0.659 > 0.579) **+13.82%**
- Recall: Empeora (0.546 < 0.576) **-5.21%**
- F1 Score: Mejora (0.597 > 0.577)
- F½ Score: Mejora (0.633 > 0.578) **+9.52%**
- G-mean: Mejora (0.600 > 0.577)

Comparación con Mix 1 (59):

- Accuracy: Constante (0.426 ~ 0.433)
- Precision: Mejora (0.659 > 0.611) **+7.86%**
- Recall: Empeora (0.546 < 0.597) **-8.54%**
- F1 Score: Constante (0.597 ~ 0.604)
- F½ Score: Mejora (0.633 > 0.608) +4.11%
- G-mean: Constante (0.600 ~ 0.604)

Mejora Accuracy, F1 y G-mean respecto a `weight_decay`=0.0001

-----
## Experiment 61
### *YOLOv8 Mid | Mix 3*

### Train

In [ ]:
# Set's maximum training time (in hours)
time: float = 1 # Depending on remaining Colab cuota
# This overrides the epochs argument, allowing training to automaticallystop after the specified duration.
# Useful for time-constrained training scenarios. (aka Colab cuotas)

In [ ]:
# Train model
model = YOLO("yolov8m.pt")
history = model.train(
    data=data,
    epochs=500,
    val=True,
    imgsz=640,
    batch=-1,
    patience=100,
    time = time,
    multi_scale=True,
    weight_decay=0.001, # Superior a default
    dropout=0.2,  # Inferior al anterior
    momentum=0.98, # Superior al anterior
)

Ultralytics 8.3.129 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/YOLO/3.5m.v4i.yolov8.640px/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.2, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.98, mosaic=1.0, multi_scale=True, name=train8, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose

train: Scanning /content/YOLO/3.5m.v4i.yolov8.640px/train/labels.cache... 270 images, 0 backgrounds, 0 corrupt: 100%|██████████| 270/270 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
AutoBatch: Computing optimal batch size for imgsz=640 at 60.0% CUDA memory utilization.
AutoBatch: CUDA:0 (Tesla T4) 14.74G total, 1.52G reserved, 0.90G allocated, 12.32G free
      Params      GFLOPs  GPU_mem (GB)  forward (ms) backward (ms)                   input                  output


    25856899       79.07         3.282         35.22         42.15        (1, 3, 640, 640)                    list
    25856899       158.1         3.630         35.55         73.86        (2, 3, 640, 640)                    list
    25856899       316.3         4.471         43.12         78.83        (4, 3, 640, 640)                    list
    25856899       632.5         6.027         79.81         134.7        (8, 3, 640, 640)                    list
    25856899        1265         9.093         150.9         259.3       (16, 3, 640, 640)                    list
AutoBatch: Using batch-size 10 for CUDA:0 9.20G/14.74G (62%) ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1778.1±630.6 MB/s, size: 89.0 KB)


train: Scanning /content/YOLO/3.5m.v4i.yolov8.640px/train/labels.cache... 270 images, 0 backgrounds, 0 corrupt: 100%|██████████| 270/270 [00:00<?, ?it/s]

...

    151/177      8.86G       1.49     0.8234      1.144        537        480: 100%|██████████| 27/27 [00:10<00:00,  2.56it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 6/6 [00:02<00:00,  2.91it/s]

                   all        108       3472       0.54      0.494      0.474       0.16
EarlyStopping: Training stopped early as no improvement observed in last 100 epochs. Best results observed at epoch 51, best model saved as best.pt.
To update EarlyStopping(patience=100) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.



151 epochs completed in 0.854 hours.
Optimizer stripped from runs/detect/train8/weights/last.pt, 52.0MB
Optimizer stripped from runs/detect/train8/weights/best.pt, 52.0MB

Validating runs/detect/train8/weights/best.pt...
Ultralytics 8.3.129 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 6/6 [00:08<00:00,  1.43s/it]


                   all        108       3472      0.552       0.51      0.512       0.18
Speed: 0.4ms preprocess, 10.7ms inference, 0.0ms loss, 12.6ms postprocess per image
Results saved to runs/detect/train8


In [ ]:
history

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b8bb4e4f2d0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [ ]:
# Show the hyperparameters set
model.trainer.validator.args

namespace(task='detect',
          mode='train',
          model='yolov8m.pt',
          data='/content/YOLO/3.5m.v4i.yolov8.640px/data.yaml',
          epochs=500,
          time=1,
          patience=100,
          batch=10,
          imgsz=640,
          save=True,
          save_period=-1,
          cache=False,
          device=None,
          workers=8,
          project=None,
          name='train8',
          exist_ok=False,
          pretrained=True,
          optimizer='auto',
          verbose=True,
          seed=0,
          deterministic=True,
          single_cls=False,
          rect=False,
          cos_lr=False,
          close_mosaic=10,
          resume=False,
          amp=True,
          fraction=1.0,
          profile=False,
          freeze=None,
          multi_scale=True,
          overlap_mask=True,
          mask_ratio=4,
          dropout=0.2,
          val=True,
          split='val',
          save_json=False,
          conf=0.001,
          iou=0.7,
    

In [ ]:
print(f"Saved into: {history.save_dir}")

Saved into: runs/detect/train8


### Validation

In [ ]:
# Load currently trained YOLO model
model = YOLO(f"/content/{history.save_dir}/weights/best.pt")

In [ ]:
# Validate the model
results = model.val(data=data,
          batch=64,
          conf=0.224, # best values found with optuna
          iou=0.57, # best values found with optuna
          verbose=True,
          save_json=True)

Ultralytics 8.3.129 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1378.1±340.3 MB/s, size: 90.9 KB)


val: Scanning /content/YOLO/3.5m.v4i.yolov8.640px/valid/labels.cache... 108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 108/108 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:06<00:00,  3.04s/it]


                   all        108       3472      0.584      0.529      0.544       0.21
Speed: 4.7ms preprocess, 22.3ms inference, 0.0ms loss, 2.4ms postprocess per image
Saving runs/detect/val7/predictions.json...
Results saved to runs/detect/val7


In [ ]:
print(f"Saved into: {results.save_dir}")

Saved into: runs/detect/val7


In [ ]:
save_json(results)

✅ JSON file stored in: runs/detect/val7


In [ ]:
matrix = gimme_metrics(results)

Total objects detected: 4617.0
Confusion matrix:
['43.34%', '24.80%']
['31.86%', '0.00%']


### Save results

In [ ]:
# Store model weights and metrics
save_on_cloud(source='/content/runs/', destination='/content/drive/MyDrive/save7/')

✅ Folder copied successfully:
   /content/runs/ 
  --> /content/drive/MyDrive/save7/


### Metrics

In [ ]:
matrix

[[2001.0, 1145.0], [1471.0, 0.0]]

In [ ]:
# Setting values from CM graph
TP = matrix[0][0]
FP = matrix[0][1]
FN = matrix[1][0]
# Confusion matrix
show_metrics(TP, FP, FN)

Total objects detected: 4617.0

Confusion matrix:
[ 43.34% , 24.80% ]
[ 31.86% , 0.00% ]

Metrics:
- Accuracy: 0.433
- Precision: 0.636
- Recall: 0.576
- F1 Score: 0.605
- F½ Score: 0.623
- G-mean: 0.605


Comparación con Reference:

- Precision: Mejora (0.636 > 0.579) **+9.84%**
- Recall: Constante (0.576 = 0.576)
- F1 Score: Mejora (0.605 > 0.577) +4.85%
- F½ Score: Mejora (0.623 > 0.578) **+7.78%**
- G-mean: Mejora (0.605 > 0.577) +4.85%

**Conclusión:** Mix 3 mostró mejoras significativas en Accuracy, Precision, F1 Score, F½ Score y G-mean en comparación con el modelo de Referencia, manteniendo el Recall constante.

Comparación con Mix 1 (59):

- Accuracy: Constante (0.433 = 0.433)
- Precision: Mejora (0.636 > 0.611) **+4.09%**
- Recall: Empeora (0.576 < 0.597) **-3.52%**
- F1 Score: Constante (0.605 ~ 0.604)
- F½ Score: Mejora (0.623 > 0.608)
- G-mean: Constante (0.605 ~0.604)

-----
## Experiment 62
### *YOLOv8 Mid | Mix 4*

### Train

In [23]:
# Set's maximum training time (in hours)
time: float = 1 # Depending on remaining Colab cuota
# This overrides the epochs argument, allowing training to automaticallystop after the specified duration.
# Useful for time-constrained training scenarios. (aka Colab cuotas)

In [24]:
# Train model
model = YOLO("yolov8m.pt")
history = model.train(
    data=data,
    epochs=500,
    val=True,
    imgsz=640,
    batch=-1,
    patience=100,
    time = time,
    multi_scale=True,
    #weight_decay=0.001, # Sin L2 reg
    dropout=0.2,  # Inferior al anterior
    momentum=0.98, # Superior al anterior
)

Ultralytics 8.3.129 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/YOLO/3.5m.v4i.yolov8.640px/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.2, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.98, mosaic=1.0, multi_scale=True, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=

100%|██████████| 755k/755k [00:00<00:00, 22.5MB/s]

Overriding model.yaml nc=80 with nc=1

                   from  n    params  module                                       arguments                     
  0                  -1  1      1392  ultralytics.nn.modules.conv.Conv             [3, 48, 3, 2]                 
  1                  -1  1     41664  ultralytics.nn.modules.conv.Conv             [48, 96, 3, 2]                
  2                  -1  2    111360  ultralytics.nn.modules.block.C2f             [96, 96, 2, True]             
  3                  -1  1    166272  ultralytics.nn.modules.conv.Conv             [96, 192, 3, 2]               
  4                  -1  4    813312  ultralytics.nn.modules.block.C2f             [192, 192, 4, True]           
  5                  -1  1    664320  ultralytics.nn.modules.conv.Conv             [192, 384, 3, 2]              
  6                  -1  4   3248640  ultralytics.nn.modules.block.C2f             [384, 384, 4, True]           
  7                  -1  1   1991808  ultralytics

 14             [-1, 4]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 15                  -1  2    517632  ultralytics.nn.modules.block.C2f             [576, 192, 2]                 
 16                  -1  1    332160  ultralytics.nn.modules.conv.Conv             [192, 192, 3, 2]              
 17            [-1, 12]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 18                  -1  2   1846272  ultralytics.nn.modules.block.C2f             [576, 384, 2]                 
 19                  -1  1   1327872  ultralytics.nn.modules.conv.Conv             [384, 384, 3, 2]              
 20             [-1, 9]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 21                  -1  2   4207104  ultralytics.nn.modules.block.C2f             [960, 576, 2]                 
 22        [15, 18, 21]  1   3776275  ultralytics.nn.modules.head.Detect           [1, [

100%|██████████| 5.35M/5.35M [00:00<00:00, 98.4MB/s]


...

    193/257      6.35G      1.367     0.7413      1.074        440        512: 100%|██████████| 16/16 [00:10<00:00,  1.55it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:02<00:00,  1.60it/s]

                   all        108       3472      0.557      0.509      0.481      0.161
EarlyStopping: Training stopped early as no improvement observed in last 100 epochs. Best results observed at epoch 93, best model saved as best.pt.
To update EarlyStopping(patience=100) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.



193 epochs completed in 0.752 hours.
Optimizer stripped from runs/detect/train/weights/last.pt, 52.0MB
Optimizer stripped from runs/detect/train/weights/best.pt, 52.0MB

Validating runs/detect/train/weights/best.pt...
Ultralytics 8.3.129 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:06<00:00,  1.50s/it]


                   all        108       3472      0.553      0.509      0.498      0.176
Speed: 0.3ms preprocess, 11.3ms inference, 0.0ms loss, 4.0ms postprocess per image
Results saved to runs/detect/train


In [25]:
history

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a8f867615d0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [26]:
# Show the hyperparameters set
model.trainer.validator.args

namespace(task='detect',
          mode='train',
          model='yolov8m.pt',
          data='/content/YOLO/3.5m.v4i.yolov8.640px/data.yaml',
          epochs=500,
          time=1,
          patience=100,
          batch=17,
          imgsz=640,
          save=True,
          save_period=-1,
          cache=False,
          device=None,
          workers=8,
          project=None,
          name='train',
          exist_ok=False,
          pretrained=True,
          optimizer='auto',
          verbose=True,
          seed=0,
          deterministic=True,
          single_cls=False,
          rect=False,
          cos_lr=False,
          close_mosaic=10,
          resume=False,
          amp=True,
          fraction=1.0,
          profile=False,
          freeze=None,
          multi_scale=True,
          overlap_mask=True,
          mask_ratio=4,
          dropout=0.2,
          val=True,
          split='val',
          save_json=False,
          conf=0.001,
          iou=0.7,
     

In [27]:
print(f"Saved into: {history.save_dir}")

Saved into: runs/detect/train


### Validation

In [28]:
# Load currently trained YOLO model
model = YOLO(f"/content/{history.save_dir}/weights/best.pt")

In [29]:
# Validate the model
results = model.val(data=data,
          batch=64,
          conf=0.224, # best values found with optuna
          iou=0.57, # best values found with optuna
          verbose=True,
          save_json=True)

Ultralytics 8.3.129 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2266.0±647.0 MB/s, size: 98.6 KB)


val: Scanning /content/YOLO/3.5m.v4i.yolov8.640px/valid/labels.cache... 108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 108/108 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:06<00:00,  3.49s/it]


                   all        108       3472      0.571      0.525      0.533      0.212
Speed: 0.6ms preprocess, 28.5ms inference, 0.0ms loss, 2.4ms postprocess per image
Saving runs/detect/val/predictions.json...
Results saved to runs/detect/val


In [30]:
print(f"Saved into: {results.save_dir}")

Saved into: runs/detect/val


In [31]:
save_json(results)

✅ JSON file stored in: runs/detect/val


In [32]:
matrix = gimme_metrics(results)

Total objects detected: 4636.0
Confusion matrix:
['43.77%', '25.11%']
['31.13%', '0.00%']


### Save results

In [33]:
# Store model weights and metrics
save_on_cloud(source='/content/runs/', destination='/content/drive/MyDrive/save8/')

✅ Folder copied successfully:
   /content/runs/ 
  --> /content/drive/MyDrive/save8/


### Metrics

In [34]:
matrix

[[2029.0, 1164.0], [1443.0, 0.0]]

In [35]:
# Setting values from CM graph
TP = matrix[0][0]
FP = matrix[0][1]
FN = matrix[1][0]
# Confusion matrix
show_metrics(TP, FP, FN)

Total objects detected: 4636.0

Confusion matrix:
[ 43.77% , 25.11% ]
[ 31.13% , 0.00% ]

Metrics:
- Accuracy: 0.438
- Precision: 0.635
- Recall: 0.584
- F1 Score: 0.609
- F½ Score: 0.625
- G-mean: 0.609


Comparación con Reference:

- Accuracy: Mejora (0.438 > 0.406) **+7.88%**
- Precision: Mejora (0.635 > 0.579) **+9.67%**
- Recall: Constante (0.584 ~ 0.576)
- F1 Score: Mejora (0.609 > 0.577)
- F½ Score: Mejora (0.625 > 0.578) **+8.13%**
- G-mean: Mejora (0.609 > 0.577)

Compararción con Mix 1 (59):

- Accuracy: Constante (0.438 ~ 0.433)
- Precision: Mejora (0.635 > 0.611)
- Recall: Empeora (0.584 < 0.597)
- F1 Score: Constante (0.609 ~ 0.604)
- F½ Score: Mejora (0.625 > 0.608)
- G-mean: Constante (0.609 ~ 0.604)

# Conclusiones

#### multi_scale → weight_decay → dropout → momentum → Mix 1/2/3/4

| Notebook        | Accuracy | Precision | Recall | F1 Score | F½ Score | G-mean |
|-----------------|----------|-----------|--------|----------|----------|--------|
| multi-scale (55)| 0.406    | 0.579     | 0.576  | 0.577    | 0.578    | 0.577  |
| Wight decay (56)| 0.419    | 0.648     | 0.542  | 0.590    | 0.623    | 0.592  |
| Dropout (57)    | 0.417    | 0.638     | 0.547  | 0.589    | 0.618    | 0.591  |
| Momentum (58)   | 0.416    | 0.644     | 0.540  | 0.587    | 0.620    | 0.589  |
| Mix 1 (59)      | 0.426    | 0.630     | 0.569  | 0.598    | 0.617    | 0.599  |
| Mix 2 (60)      | 0.433    | 0.611     | 0.597  | 0.604    | 0.608    | 0.604  |
| Mix 3 (61)      | 0.426    | 0.659     | 0.546  | 0.597    | 0.633    | 0.600  |
| Mix 4 (62)      | 0.433    | 0.636     | 0.576  | 0.605    | 0.623    | 0.605  |

### Multi Scale

Este hiperparámetro  parece ser fundamental para obtener un mejor rendimiento general, ya que fue la métrica que mayor impacto positivo generó.

- Comparando la "Referencia" con el Exp 55, vemos que activar multi_scale resultó en mejoras en Accuracy, Precision, F1 Score, F½ Score y G-mean, aunque con una disminución en Recall.

### Weight Decay (L2 regularization)

- Comparando "Exp 55" (weight_decay=0.0005, momentum=0.937, dropout=0) con "Mix 2" (weight_decay=0.001, momentum=0.98, dropout=0), el aumento en weight_decay (junto con el aumento en momentum) parece contribuir a una mejora significativa en Precision y F½ Score, aunque el Recall se mantiene bajo.
- Comparando "Dropout (57)" (weight_decay=0.0005, momentum=0.937, dropout=0.2) con "Mix 3" (weight_decay=0.001, momentum=0.98, dropout=0.2), el aumento en weight_decay (junto con el aumento en momentum) en Mix 3 resultó en mejoras en casi todas las métricas, incluyendo un Recall que se mantuvo en el nivel de la Referencia, algo que Dropout (57) no logró.

Esto sugiere que aumentar el weight_decay, especialmente en combinación con un momentum alto, puede ser beneficioso para mejorar la regularización y el rendimiento en métricas como Precision y F½ Score, y puede ayudar a mitigar la caída de Recall observada con dropout en otros contextos.

### Dropout
El impacto del dropout parece depender fuertemente de la interacción con otros hiperparámetros.

- Un dropout=0.2 por sí solo (Exp 57) perjudicó el Recall.

- Sin embargo, un dropout=0.2 combinado con weight_decay=0.001 y momentum=0.98 (Mix 3) logra mantener un buen Recall mientras impulsa Accuracy, F1 Score y G-mean a los valores más altos.

- Un dropout=0.4 (Mix 1), combinado con momentum=0.95 y weight_decay=0.0005, maximiza Recall pero reduce Precision y F½ Score.

- Mantener dropout=0 (Mix 2), combinado con weight_decay=0.001 y momentum=0.98, maximiza Precision y F½ Score pero resulta en bajo Recall.

En este contexto, el valor de dropout=0.2 en combinación con weight_decay=0.001 y momentum=0.98 (Mix 3) parece ofrecer el mejor equilibrio general si se buscan altas métricas compuestas (F1, G-mean) y un buen Recall, sin sacrificar demasiado Precision y F½ Score. Si la prioridad absoluta es F½ Score y Precision, entonces dropout=0 en combinación con weight_decay=0.001 y momentum=0.98 (Mix 2) es superior, aunque con menor Recall.

### Momentum


- Comparando "Exp 55" (momentum=0.937, dropout=0, weight_decay=0.0005) con "Momentum (58)" (momentum=0.95, mismos otros parámetros), un ligero aumento en momentum mejoró Accuracy, Precision, F1 Score, F½ Score y G-mean, manteniendo Recall constante.
- Comparando "Exp 55" con "Mix 2" (momentum=0.98, weight_decay=0.001, dropout=0), un aumento más significativo en momentum (junto con weight decay) llevó a mejoras aún mayores en Precision y F½ Score.
- Comparando "Dropout (57)" (momentum=0.937, weight_decay=0.0005, dropout=0.2) con "Mix 3" (momentum=0.98, weight_decay=0.001, dropout=0.2), el aumento en momentum (junto con weight decay) resultó en un rendimiento superior en Mix 3 en la mayoría de las métricas.

Esto indica que aumentar el momentum, especialmente a valores como 0.98, parece ser un factor clave para mejorar la convergencia y el rendimiento final del modelo.

## Mejores modelos

Comparando el F½ Score, el experimento Mix 2 obtuvo el mejor resultado (0.633).

Además, Mix 2 también obtuvo la mejor Precision (0.659) entre estos tres experimentos.
- Previously best (Exp 55): F½ Score = 0.623, Precision = 0.648
- Mix 1 (59): F½ Score = 0.608, Precision = 0.611
- Mix 2: F½ Score = 0.633, Precision = 0.659

| Notebook          | Accuracy | Precision | Recall | F1 Score | F½ Score | G-mean |
| :---------------- | :------- | :-------- | :----- | :------- | :------- | :----- |
| referencia        | 0.406    | 0.579     | 0.576  | 0.577    | 0.578    | 0.577  |
| multi-scale (55)  | 0.419    | 0.648     | 0.542  | 0.590    | 0.623    | 0.592  |
| Mix 1 (59)        | 0.433    | 0.611     | 0.597  | 0.604    | 0.608    | 0.604  |
| Mix 2  (60) | 0.426    | 0.659     | 0.546  | 0.597    | 0.633    | 0.600  |

# Estrategias a futuro

Se explorarán distintos mezclas de hiperparámetros para ver si es posible mejorar los resultados obtenidos con el Mix 2 y 3.

### Experimento A:
- multi_scale=True
- weight_decay=0.0015 (Explorar un - weight_decay ligeramente mayor)
- momentum=0.98
- dropout=0 (default)

**Justificación:** Mantener los parámetros exitosos de Mix 2 y ver si una regularización de weight_decay un poco más fuerte mejora aún más el rendimiento, especialmente en Precision y F½ Score.

### Experimento B:
- multi_scale=True
- weight_decay=0.001
- momentum=0.99 (Explorar un momentum aún más cercano a 1)
- dropout=0 (default)

**Justificación:** Mantener el weight_decay de Mix 2 y probar si un momentum extremadamente alto puede ofrecer beneficios adicionales para la convergencia y el rendimiento final en las métricas objetivo.


### Experimento C:
Best mix + dropout